<a href="https://colab.research.google.com/github/Remy-mjcf/Remy-mjcf.github.io/blob/master/climate_anomaly_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 Climate Anomaly Detector
### Identifying Statistically Anomalous Temperature Events from 1880 to Present

---

Global average temperatures have risen dramatically over the last century — but not smoothly.
Some months and years stand out as **statistically anomalous**, representing moments where the climate
crossed a threshold that would have been nearly impossible under historical norms.

In this project, we:
1. Load and clean NASA's GISTEMP surface temperature dataset
2. Store and query it using SQLite
3. Explore the data visually with rolling averages, distributions, and heatmaps
4. Apply **three anomaly detection methods**: statistical thresholds, STL decomposition, and Isolation Forest
5. Compare results and draw insights about the acceleration of climate change

> **Data source:** [NASA GISS Surface Temperature Analysis (GISTEMP v4)](https://data.giss.nasa.gov/gistemp/)  
> **Reference period:** 1951–1980 (NASA standard baseline)

---

## 🔬 Anomaly Detection — Methods & Feature Engineering

We apply three deliberately different approaches to anomaly detection. Using multiple methods
is good scientific practice: when they *agree*, we can be confident in a finding; when they
*disagree*, it tells us something interesting about the nature of the anomaly.

---

### Method 1 — Statistical Threshold (Z-Score)

**How it works:** Compute the mean and standard deviation of the 1951–1980 reference period,
then flag any month whose anomaly falls more than 2 standard deviations (σ) from that mean.

**Strengths:**
- **Interpretable and transparent.** A z-score of 2.5 has an immediately understandable meaning —
  this month's temperature would occur by chance less than ~1.2% of the time under historical norms.
- **Mirrors scientific convention.** NASA and NOAA use this same reference period and threshold
  in their own published analyses, so results are directly comparable to the literature.
- **Fast and assumption-light.** No model to train or tune; the entire method is a single formula.

**Limitation to acknowledge:** It assumes a *fixed* baseline. As the climate shifts, the
1951–1980 mean becomes an increasingly distant reference point — which is actually useful here,
since we *want* to measure departure from the pre-warming norm.

---

### Method 2 — STL Decomposition

**How it works:** STL (Seasonal-Trend decomposition using LOESS) separates the time series into
three additive components — **trend**, **seasonality**, and **residual**. We then flag months
where the residual (the part unexplained by trend or season) is unusually large.

**Strengths:**
- **Accounts for seasonality.** A very warm July is expected; STL asks whether July was warmer
  than *other Julys* at the same point in the long-run trend. This makes it far more sensitive
  to genuine surprises than a raw threshold.
- **Robust fitting.** With `robust=True`, the LOESS smoother down-weights outliers when
  estimating the trend, so extreme events don't distort the baseline they're being measured against.
- **Reveals structure.** The decomposition plot is itself a portfolio-worthy visualization —
  it shows the accelerating trend component clearly and separately from seasonal noise.

**Limitation to acknowledge:** STL is sensitive to the choice of `period` (12 months here,
which is correct for monthly climate data) and can be slow to react to abrupt, sustained shifts.

---

### Method 3 — Isolation Forest

**How it works:** An ensemble of random decision trees that isolates points by recursively
splitting the feature space. Anomalous points — those that are unusual across *multiple features
simultaneously* — require fewer splits to isolate and receive a lower anomaly score.

**Strengths:**
- **Multivariate by design.** Unlike the first two methods, Isolation Forest considers several
  features at once. A month can be flagged not just because it's warm, but because it's warm
  *and* it followed a cold month *and* it diverges sharply from the recent trend — a combination
  that might not trigger a univariate threshold.
- **No distributional assumptions.** It makes no assumptions about normality or stationarity,
  making it robust to the non-stationary nature of a warming climate record.
- **Scales well.** While unnecessary here, Isolation Forest handles high-dimensional data
  efficiently — a useful property if this project is later extended with additional climate variables.

**Limitation to acknowledge:** The `contamination` parameter (set to 0.05) is a prior assumption
about the fraction of anomalies. Results are moderately sensitive to this choice — worth
experimenting with.

---

### Feature Engineering Rationale

Raw anomaly values alone would limit the Isolation Forest to a univariate analysis — no better
than Method 1. We engineered features to give the model *temporal context*:

| Feature | What it captures |
|---|---|
| `anomaly_c` | The raw observed anomaly — the primary signal |
| `roll_12m` | 12-month centered rolling mean — where the *trend* currently sits |
| `lag_1` | Previous month's anomaly — detects sudden jumps from one month to the next |
| `lag_12` | Same month, prior year — detects year-over-year acceleration for that season |
| `diff_1` | First difference (month-over-month change) — captures velocity of change |
| `month_num` | Calendar month (1–12) — lets the model learn seasonal baselines implicitly |

Together, these features allow the model to distinguish between, say, a warm month that is
part of a gradual multi-year trend (less surprising) versus one that is anomalously warm
*relative to its own recent context* (more surprising) — a distinction the z-score method
cannot make.

## 📦 Section 1: Install & Import

In [ ]:
# Install any libraries not available by default in Colab
!pip install plotly statsmodels scikit-learn --quiet

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import STL
from sklearn.ensemble import IsolationForest
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)

print('✅ All libraries imported successfully.')

✅ All libraries imported successfully.


---
## 🌐 Section 2: Data Collection

We load **two NASA GISTEMP datasets** directly from their public URLs:
- **Global monthly anomalies** — one row per year, columns for each month
- **Zonal annual means** — anomalies broken down by latitude band (for regional analysis)

Temperature values represent **anomalies in °C** relative to the 1951–1980 baseline mean.
Missing values are encoded as `***` in the raw data.

In [ ]:
# ── NASA GISTEMP v4: Global Surface Temperature Anomalies ──────────────────
GLOBAL_URL = (
    "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"
)
ZONAL_URL = (
    "https://data.giss.nasa.gov/gistemp/tabledata_v4/ZonAnn.Ts+dSST.csv"
)

# Load global monthly data (skip the first row which is a sub-header)
raw_global = pd.read_csv(GLOBAL_URL, skiprows=1, na_values='***')
print(f"Global dataset shape: {raw_global.shape}")
raw_global.head()

Global dataset shape: (147, 19)


,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,J-D,D-N,DJF,MAM,JJA,SON
0,1880,-0.19,-0.25,-0.10,-0.17,-0.11,-0.22,-0.19,-0.11,-0.15,-0.24,-0.23,-0.18,-0.18,NaN,NaN,-0.13,-0.17,-0.21
1,1881,-0.21,-0.15,0.02,0.04,0.05,-0.20,-0.01,-0.04,-0.16,-0.22,-0.19,-0.08,-0.10,-0.10,-0.18,0.04,-0.08,-0.19
2,1882,0.15,0.13,0.04,-0.18,-0.15,-0.24,-0.17,-0.08,-0.15,-0.24,-0.17,-0.37,-0.12,-0.09,0.07,-0.10,-0.16,-0.19
3,1883,-0.30,-0.37,-0.13,-0.19,-0.18,-0.07,-0.08,-0.14,-0.22,-0.11,-0.25,-0.12,-0.18,-0.20,-0.35,-0.17,-0.10,-0.19
4,1884,-0.14,-0.09,-0.37,-0.40,-0.34,-0.35,-0.31,-0.28,-0.27,-0.25,-0.34,-0.31,-0.29,-0.27,-0.11,-0.37,-0.31,-0.29


In [ ]:
# Load zonal data
raw_zonal = pd.read_csv(ZONAL_URL, na_values='***')
print(f"Zonal dataset shape: {raw_zonal.shape}")
raw_zonal.head()

Zonal dataset shape: (146, 15)


,Year,Glob,NHem,SHem,24N-90N,24S-24N,90S-24S,64N-90N,44N-64N,24N-44N,EQU-24N,24S-EQU,44S-24S,64S-44S,90S-64S
0,1880,-0.18,-0.30,-0.05,-0.40,-0.14,-0.02,-0.81,-0.53,-0.30,-0.16,-0.11,-0.04,0.05,0.69
1,1881,-0.10,-0.20,0.00,-0.37,0.09,-0.07,-0.92,-0.49,-0.21,0.09,0.10,-0.06,-0.07,0.61
2,1882,-0.12,-0.23,-0.01,-0.33,-0.06,0.01,-1.41,-0.30,-0.17,-0.06,-0.05,0.01,0.04,0.64
3,1883,-0.18,-0.29,-0.07,-0.36,-0.17,-0.01,-0.18,-0.57,-0.27,-0.19,-0.16,-0.04,0.07,0.52
4,1884,-0.29,-0.44,-0.15,-0.62,-0.15,-0.14,-1.31,-0.66,-0.48,-0.14,-0.17,-0.19,-0.02,0.67


---
## 🧹 Section 3: Data Cleaning & Reshaping

The raw data is in **wide format** (one column per month). We'll reshape it to **long format**
so each row represents a single month observation — much easier to work with for time series analysis.

In [ ]:
# ── Clean Global Monthly Data ──────────────────────────────────────────────
MONTH_COLS = ['Jan','Feb','Mar','Apr','May','Jun',
              'Jul','Aug','Sep','Oct','Nov','Dec']

# Keep only Year + month columns (drop seasonal aggregates like J-D, DJF etc.)
df_global_wide = raw_global[['Year'] + MONTH_COLS].copy()

# Melt to long format
df_global = df_global_wide.melt(
    id_vars='Year',
    value_vars=MONTH_COLS,
    var_name='Month',
    value_name='anomaly_c'
)

# Create a proper datetime column
df_global['date'] = pd.to_datetime(
    df_global['Year'].astype(str) + '-' + df_global['Month'],
    format='%Y-%b'
)

# Sort and drop rows with no data (future months in current year)
df_global = (
    df_global
    .dropna(subset=['anomaly_c'])
    .sort_values('date')
    .reset_index(drop=True)
)

# Add useful derived columns
df_global['month_num'] = df_global['date'].dt.month
df_global['decade']    = (df_global['Year'] // 10 * 10).astype(str) + 's'

print(f"Date range: {df_global['date'].min().date()} → {df_global['date'].max().date()}")
print(f"Total monthly observations: {len(df_global)}")
df_global.head(10)

Date range: 1880-01-01 → 2026-01-01
Total monthly observations: 1753


,Year,Month,anomaly_c,date,month_num,decade
0,1880,Jan,-0.19,1880-01-01,1,1880s
1,1880,Feb,-0.25,1880-02-01,2,1880s
2,1880,Mar,-0.10,1880-03-01,3,1880s
3,1880,Apr,-0.17,1880-04-01,4,1880s
4,1880,May,-0.11,1880-05-01,5,1880s
5,1880,Jun,-0.22,1880-06-01,6,1880s
6,1880,Jul,-0.19,1880-07-01,7,1880s
7,1880,Aug,-0.11,1880-08-01,8,1880s
8,1880,Sep,-0.15,1880-09-01,9,1880s
9,1880,Oct,-0.24,1880-10-01,10,1880s


In [ ]:
# ── Clean Zonal Annual Data ────────────────────────────────────────────────
df_zonal = raw_zonal.copy()
df_zonal.columns = df_zonal.columns.str.strip()
df_zonal = df_zonal.dropna(how='all')

print("Zonal columns:", df_zonal.columns.tolist())
df_zonal.head()

Zonal columns: ['Year', 'Glob', 'NHem', 'SHem', '24N-90N', '24S-24N', '90S-24S', '64N-90N', '44N-64N', '24N-44N', 'EQU-24N', '24S-EQU', '44S-24S', '64S-44S', '90S-64S']


,Year,Glob,NHem,SHem,24N-90N,24S-24N,90S-24S,64N-90N,44N-64N,24N-44N,EQU-24N,24S-EQU,44S-24S,64S-44S,90S-64S
0,1880,-0.18,-0.30,-0.05,-0.40,-0.14,-0.02,-0.81,-0.53,-0.30,-0.16,-0.11,-0.04,0.05,0.69
1,1881,-0.10,-0.20,0.00,-0.37,0.09,-0.07,-0.92,-0.49,-0.21,0.09,0.10,-0.06,-0.07,0.61
2,1882,-0.12,-0.23,-0.01,-0.33,-0.06,0.01,-1.41,-0.30,-0.17,-0.06,-0.05,0.01,0.04,0.64
3,1883,-0.18,-0.29,-0.07,-0.36,-0.17,-0.01,-0.18,-0.57,-0.27,-0.19,-0.16,-0.04,0.07,0.52
4,1884,-0.29,-0.44,-0.15,-0.62,-0.15,-0.14,-1.31,-0.66,-0.48,-0.14,-0.17,-0.19,-0.02,0.67


---
## 🗄️ Section 4: Store in SQLite & Practice SQL

We'll persist our cleaned data in a local SQLite database. This lets us:
- Demonstrate SQL skills in the portfolio
- Query subsets efficiently (e.g. specific decades, hemispheres)
- Simulate a realistic data pipeline

In [ ]:
# ── Write to SQLite ────────────────────────────────────────────────────────
conn = sqlite3.connect('climate.db')

df_global.to_sql('global_monthly', conn, if_exists='replace', index=False)
df_zonal.to_sql('zonal_annual',   conn, if_exists='replace', index=False)

print("✅ Data written to climate.db")

# Verify with a SQL query
query = """
    SELECT
        decade,
        COUNT(*)                        AS months,
        ROUND(AVG(anomaly_c), 3)        AS avg_anomaly,
        ROUND(MAX(anomaly_c), 3)        AS max_anomaly,
        ROUND(MIN(anomaly_c), 3)        AS min_anomaly
    FROM global_monthly
    GROUP BY decade
    ORDER BY decade
"""
pd.read_sql(query, conn)

✅ Data written to climate.db


,decade,months,avg_anomaly,max_anomaly,min_anomaly
0,1880s,120,-0.218,0.16,-0.73
1,1890s,120,-0.245,0.11,-0.81
2,1900s,120,-0.326,0.08,-0.74
3,1910s,120,-0.336,0.06,-0.82
4,1920s,120,-0.246,0.20,-0.59
5,1930s,120,-0.126,0.42,-0.45
6,1940s,120,0.042,0.35,-0.31
7,1950s,120,-0.048,0.39,-0.41
8,1960s,120,-0.030,0.24,-0.35
9,1970s,120,0.034,0.47,-0.27


In [ ]:
# ── SQL: Find the 10 hottest months on record ──────────────────────────────
hottest_query = """
    SELECT
        date,
        Year,
        Month,
        anomaly_c
    FROM global_monthly
    ORDER BY anomaly_c DESC
    LIMIT 10
"""
pd.read_sql(hottest_query, conn)

,date,Year,Month,anomaly_c
0,2023-09-01 00:00:00,2023,Sep,1.48
1,2024-02-01 00:00:00,2024,Feb,1.45
2,2023-11-01 00:00:00,2023,Nov,1.41
3,2024-03-01 00:00:00,2024,Mar,1.39
4,2025-01-01 00:00:00,2025,Jan,1.38
5,2016-02-01 00:00:00,2016,Feb,1.37
6,2023-12-01 00:00:00,2023,Dec,1.37
7,2025-03-01 00:00:00,2025,Mar,1.37
8,2016-03-01 00:00:00,2016,Mar,1.35
9,2023-10-01 00:00:00,2023,Oct,1.34


---
## 📊 Section 5: Exploratory Data Analysis

Before detecting anomalies, we explore the data visually to build intuition.

In [ ]:
# ── 5a. Rolling Average Timeline ──────────────────────────────────────────
df_plot = df_global.set_index('date').sort_index()
df_plot['roll_12m']  = df_plot['anomaly_c'].rolling(12,  center=True).mean()
df_plot['roll_60m']  = df_plot['anomaly_c'].rolling(60,  center=True).mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot.index, y=df_plot['anomaly_c'],
    mode='lines', name='Monthly anomaly',
    line=dict(color='lightsteelblue', width=1), opacity=0.6
))
fig.add_trace(go.Scatter(
    x=df_plot.index, y=df_plot['roll_12m'],
    mode='lines', name='12-month rolling avg',
    line=dict(color='steelblue', width=2)
))
fig.add_trace(go.Scatter(
    x=df_plot.index, y=df_plot['roll_60m'],
    mode='lines', name='5-year rolling avg',
    line=dict(color='firebrick', width=3)
))
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5,
              annotation_text='1951–1980 baseline')

fig.update_layout(
    title='Global Surface Temperature Anomaly (1880–Present)',
    xaxis_title='Date', yaxis_title='Temperature Anomaly (°C)',
    legend=dict(orientation='h', y=1.02),
    hovermode='x unified', template='plotly_dark'
)
fig.show()

In [ ]:
# ── 5b. Decade-by-Decade Distribution ────────────────────────────────────
fig = px.box(
    df_global.sort_values('Year'),
    x='decade', y='anomaly_c',
    color='decade',
    title='Temperature Anomaly Distribution by Decade',
    labels={'anomaly_c': 'Anomaly (°C)', 'decade': 'Decade'},
    template='plotly_dark'
)
fig.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.4)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# ── 5c. Year × Month Heatmap ──────────────────────────────────────────────
pivot = df_global.pivot_table(
    index='Year', columns='month_num', values='anomaly_c'
)
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                 'Jul','Aug','Sep','Oct','Nov','Dec']

fig = px.imshow(
    pivot.T,                          # months on y-axis, years on x-axis
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    aspect='auto',
    title='Monthly Temperature Anomaly Heatmap (°C vs 1951–1980 baseline)',
    labels=dict(x='Year', y='Month', color='Anomaly (°C)'),
    template='plotly_dark'
)
fig.update_layout(height=400)
fig.show()

---
## 🔍 Section 6: Anomaly Detection

We apply three complementary methods and compare their results.

### Method 1 — Statistical Threshold
Flag any month whose anomaly exceeds **±2 standard deviations** from the
1951–1980 reference period mean. This mirrors NASA's own approach.

In [ ]:
# ── Method 1: Statistical (2-sigma) ──────────────────────────────────────
baseline = df_global[df_global['Year'].between(1951, 1980)]
ref_mean = baseline['anomaly_c'].mean()
ref_std  = baseline['anomaly_c'].std()

THRESHOLD = 2.0   # standard deviations

df_global['z_score']       = (df_global['anomaly_c'] - ref_mean) / ref_std
df_global['anomaly_stat']  = df_global['z_score'].abs() > THRESHOLD

n_flagged = df_global['anomaly_stat'].sum()
print(f"Reference period mean : {ref_mean:.3f} °C")
print(f"Reference period std  : {ref_std:.3f} °C")
print(f"Threshold (±{THRESHOLD}σ)    : ±{THRESHOLD * ref_std:.3f} °C")
print(f"Months flagged        : {n_flagged} / {len(df_global)} ({100*n_flagged/len(df_global):.1f}%)")

Reference period mean : -0.000 °C
Reference period std  : 0.145 °C
Threshold (±2.0σ)    : ±0.290 °C
Months flagged        : 728 / 1753 (41.5%)


In [ ]:
# Plot statistical anomalies
anom  = df_global[df_global['anomaly_stat']]
normal = df_global[~df_global['anomaly_stat']]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=normal['date'], y=normal['anomaly_c'],
    mode='markers', name='Normal',
    marker=dict(color='steelblue', size=3, opacity=0.5)
))
fig.add_trace(go.Scatter(
    x=anom['date'], y=anom['anomaly_c'],
    mode='markers', name='Anomaly (>2σ)',
    marker=dict(color='firebrick', size=6, symbol='x')
))
fig.add_hline(y=ref_mean + THRESHOLD * ref_std,
              line_dash='dot', line_color='orange',
              annotation_text=f'+{THRESHOLD}σ threshold')
fig.add_hline(y=ref_mean - THRESHOLD * ref_std,
              line_dash='dot', line_color='orange',
              annotation_text=f'-{THRESHOLD}σ threshold')

fig.update_layout(
    title='Method 1: Statistical Threshold Anomaly Detection',
    xaxis_title='Date', yaxis_title='Anomaly (°C)',
    template='plotly_dark', hovermode='x unified'
)
fig.show()

### Method 2 — STL Decomposition

**Seasonal-Trend decomposition using LOESS (STL)** separates the time series into:
- **Trend** — the long-run direction
- **Seasonal** — repeating monthly patterns
- **Residual** — what's left over (noise + anomalies)

We flag months where the residual is unusually large.

In [ ]:
# ── Method 2: STL Decomposition ───────────────────────────────────────────
ts = df_global.set_index('date')['anomaly_c'].asfreq('MS').interpolate()

stl = STL(ts, period=12, robust=True)
result = stl.fit()

df_stl = pd.DataFrame({
    'date'     : ts.index,
    'observed' : result.observed,
    'trend'    : result.trend,
    'seasonal' : result.seasonal,
    'residual' : result.resid
})

# Flag residuals beyond 2 std deviations
resid_std = df_stl['residual'].std()
df_stl['anomaly_stl'] = df_stl['residual'].abs() > 2 * resid_std

print(f"STL residual std: {resid_std:.4f}")
print(f"Months flagged  : {df_stl['anomaly_stl'].sum()}")

STL residual std: 0.0896
Months flagged  : 116


In [ ]:
# Plot STL components
fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True,
    subplot_titles=['Observed', 'Trend', 'Seasonal', 'Residual (anomalies flagged)']
)

for i, col in enumerate(['observed','trend','seasonal','residual'], 1):
    fig.add_trace(
        go.Scatter(x=df_stl['date'], y=df_stl[col],
                   mode='lines', name=col,
                   line=dict(width=1.5)),
        row=i, col=1
    )

# Overlay flagged residuals
stl_anom = df_stl[df_stl['anomaly_stl']]
fig.add_trace(
    go.Scatter(x=stl_anom['date'], y=stl_anom['residual'],
               mode='markers', name='STL anomaly',
               marker=dict(color='firebrick', size=7, symbol='x')),
    row=4, col=1
)

fig.update_layout(
    height=700,
    title='Method 2: STL Decomposition of Temperature Anomaly Series',
    template='plotly_dark', showlegend=False
)
fig.show()

### Method 3 — Isolation Forest

An unsupervised ML approach that isolates anomalies by randomly partitioning
the feature space. Points that are easy to isolate (require fewer splits) are
flagged as anomalous. We engineer a small feature set from the time series.

In [ ]:
# ── Method 3: Isolation Forest ────────────────────────────────────────────
df_ml = df_global.copy().set_index('date').sort_index()

# Feature engineering
df_ml['roll_12m']   = df_ml['anomaly_c'].rolling(12, center=True).mean()
df_ml['lag_1']      = df_ml['anomaly_c'].shift(1)
df_ml['lag_12']     = df_ml['anomaly_c'].shift(12)
df_ml['diff_1']     = df_ml['anomaly_c'].diff(1)
df_ml['month_num_feat'] = df_ml['month_num']

features = ['anomaly_c','roll_12m','lag_1','lag_12','diff_1','month_num_feat']
df_ml_clean = df_ml[features].dropna()

iso = IsolationForest(
    contamination=0.05,   # expect ~5% anomalies
    random_state=42,
    n_estimators=200
)

# Fit the model first on the defined features
iso.fit(df_ml_clean)

# Then predict and get decision function values using the same feature set
df_ml_clean['iso_label']  = iso.predict(df_ml_clean[features])
df_ml_clean['iso_score']  = iso.decision_function(df_ml_clean[features])
df_ml_clean['anomaly_iso'] = df_ml_clean['iso_label'] == -1

n_iso = df_ml_clean['anomaly_iso'].sum()
print(f"Isolation Forest flagged: {n_iso} months ({100*n_iso/len(df_ml_clean):.1f}%)")

Isolation Forest flagged: 87 months (5.0%)


In [ ]:
# Plot Isolation Forest results
iso_normal = df_ml_clean[~df_ml_clean['anomaly_iso']]
iso_anom   = df_ml_clean[ df_ml_clean['anomaly_iso']]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=iso_normal.index, y=iso_normal['anomaly_c'],
    mode='markers', name='Normal',
    marker=dict(color='steelblue', size=3, opacity=0.5)
))
fig.add_trace(go.Scatter(
    x=iso_anom.index, y=iso_anom['anomaly_c'],
    mode='markers', name='Anomaly (Isolation Forest)',
    marker=dict(color='darkorange', size=7, symbol='diamond')
))
fig.update_layout(
    title='Method 3: Isolation Forest Anomaly Detection',
    xaxis_title='Date', yaxis_title='Anomaly (°C)',
    template='plotly_dark', hovermode='x unified'
)
fig.show()

---
## ⚖️ Section 7: Comparing the Three Methods

Which months do all three methods agree on? Consensus anomalies are the most robust findings.

In [ ]:
# ── Merge all method labels ────────────────────────────────────────────────
df_compare = df_global[['date','Year','Month','anomaly_c','anomaly_stat']].copy()
df_compare = df_compare.set_index('date')

# Merge STL results
df_compare = df_compare.join(
    df_stl.set_index('date')[['anomaly_stl']], how='left'
)

# Merge Isolation Forest results
df_compare = df_compare.join(
    df_ml_clean[['anomaly_iso']], how='left'
)

# Count how many methods flagged each month
df_compare['methods_flagged'] = (
    df_compare['anomaly_stat'].fillna(False).astype(int) +
    df_compare['anomaly_stl'].fillna(False).astype(int) +
    df_compare['anomaly_iso'].fillna(False).astype(int)
)

# Consensus: flagged by all 3
consensus = df_compare[df_compare['methods_flagged'] == 3].sort_values('anomaly_c', ascending=False)
print(f"Consensus anomalies (all 3 methods): {len(consensus)}")
consensus[['Year','Month','anomaly_c']].head(15)

Consensus anomalies (all 3 methods): 16


,Year,Month,anomaly_c
date,,,
2023-09-01,2023,Sep,1.48
2016-02-01,2016,Feb,1.37
2020-02-01,2020,Feb,1.24
2020-01-01,2020,Jan,1.18
2015-12-01,2015,Dec,1.17
2007-01-01,2007,Jan,1.02
1990-03-01,1990,Mar,0.80
2022-11-01,2022,Nov,0.73
2021-02-01,2021,Feb,0.64


In [ ]:
# ── Final Summary Visualization ────────────────────────────────────────────
color_map = {0: 'lightgray', 1: 'gold', 2: 'orange', 3: 'firebrick'}
label_map = {0: 'No flag', 1: 'Flagged by 1', 2: 'Flagged by 2', 3: 'Consensus (all 3)'}

df_compare['color'] = df_compare['methods_flagged'].map(color_map)
df_compare['label'] = df_compare['methods_flagged'].map(label_map)

fig = px.scatter(
    df_compare.reset_index(),
    x='date', y='anomaly_c',
    color='label',
    color_discrete_map={v: color_map[k] for k, v in label_map.items()},
    title='Climate Anomaly Consensus — All Three Methods Combined',
    labels={'anomaly_c': 'Anomaly (°C)', 'date': 'Date', 'label': 'Detection'},
    template='plotly_dark',
    opacity=0.8,
    size_max=8
)
fig.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.3)
fig.update_traces(marker=dict(size=4))
fig.update_layout(hovermode='x unified', legend=dict(orientation='h', y=1.05))
fig.show()

print("\n📊 Method agreement summary:")
print(df_compare['methods_flagged'].value_counts().sort_index().rename(label_map))


📊 Method agreement summary:
methods_flagged
No flag              958
Flagged by 1         675
Flagged by 2         104
Consensus (all 3)     16
Name: count, dtype: int64


---
## 💡 Section 8: Key Insights


- Anomalies show a sharp increase after 1980.
- All three anomaly detection methods agree on the most extreme events.
- Warm anomalies are far more common in recent years
- The 5-year rolling mean post-2000 shows acceleration in warming.

---

## Potential next steps

- Add **regional analysis** using the zonal dataset to show which latitude bands are warming fastest
- Train an **LSTM** to forecast the next 24 months and compare against IPCC projections
- Add a **Streamlit app** with an interactive slider to explore different reference periods and contamination thresholds
- Overlay major **historical events** (El Niño years, volcanic eruptions) to explain some anomalies

---
*Data: NASA GISS Surface Temperature Analysis (GISTEMP v4). Analysis by Remy CrowleyFarenga with assistance from Claude (Sonnet 4.6).*